# 05 — Model Comparison & Best Model Selection

Compare all 4 trained models using a weighted composite score:

| Metric | Weight |
|--------|--------|
| Macro F1 | 50% |
| Top-1 Accuracy | 20% |
| Inference Latency | 15% |
| Model Size | 10% |
| Calibration | 5% |

### Expected Output
- Radar chart comparing all models
- Grouped bar chart of key metrics
- Comparison table with rankings
- Best model recommendation

In [ ]:
import sys, os, json
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if 'notebooks' in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from ml.src.evaluation.compare_models import (
    compute_weighted_scores,
    plot_comparison_radar,
    plot_comparison_bar,
    generate_comparison_report,
)

## 1. Load All Model Reports

In [ ]:
reports_dir = PROJECT_ROOT / 'ml' / 'artifacts' / 'reports'
model_names = ['mlp', 'cnn', 'resnet', 'vit']

model_reports = []
for name in model_names:
    report_path = reports_dir / f'{name}_report.json'
    if report_path.exists():
        with open(report_path) as f:
            report = json.load(f)
        model_reports.append(report)
        print(f'✓ Loaded {name} report')
    else:
        print(f'✗ Missing {name} report — run notebook {model_names.index(name)+1:02d} first')

print(f'\nLoaded {len(model_reports)} / {len(model_names)} model reports')

## 2. Summary Table

In [ ]:
rows = []
for r in model_reports:
    metrics = r.get('metrics', {})
    latency = r.get('latency', {})
    rows.append({
        'Model': r['model_name'],
        'Accuracy': f"{metrics.get('accuracy', 0):.4f}",
        'Macro F1': f"{metrics.get('macro_f1', 0):.4f}",
        'Macro Precision': f"{metrics.get('macro_precision', 0):.4f}",
        'Macro Recall': f"{metrics.get('macro_recall', 0):.4f}",
        'Latency (ms)': f"{latency.get('avg_ms', 0):.1f}",
        'Size (MB)': f"{r.get('model_size_mb', 0):.1f}",
        'Parameters': f"{r.get('num_parameters', 0):,}",
    })

df_summary = pd.DataFrame(rows)
df_summary.style.set_caption('Model Comparison Summary')

## 3. Weighted Scoring & Ranking

In [ ]:
figures_dir = PROJECT_ROOT / 'ml' / 'artifacts' / 'figures' / 'comparison'
figures_dir.mkdir(parents=True, exist_ok=True)

comparison_report = generate_comparison_report(
    model_reports,
    output_dir=figures_dir,
)

print('\n=== Rankings ===')
for r in comparison_report['rankings']:
    print(f"  #{r['rank']} {r['model_name']:>8s}  "
          f"composite={r['composite_score']:.4f}  "
          f"F1={r['raw_metrics']['macro_f1']:.4f}  "
          f"Acc={r['raw_metrics']['accuracy']:.4f}  "
          f"Lat={r['raw_metrics']['latency_ms']:.1f}ms  "
          f"Size={r['raw_metrics']['model_size_mb']:.1f}MB")

## 4. Radar Chart

In [ ]:
from IPython.display import Image as IPImage, display
radar_path = figures_dir / 'comparison_radar.png'
if radar_path.exists():
    display(IPImage(filename=str(radar_path), width=600))

## 5. Bar Chart

In [ ]:
bar_path = figures_dir / 'comparison_bar.png'
if bar_path.exists():
    display(IPImage(filename=str(bar_path), width=800))

## 6. Per-Model Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(model_reports), figsize=(6*len(model_reports), 5))
if len(model_reports) == 1:
    axes = [axes]

for idx, report in enumerate(model_reports):
    cm_path = PROJECT_ROOT / 'ml' / 'artifacts' / 'figures' / report['model_name'] / 'confusion_matrix.png'
    if cm_path.exists():
        img = plt.imread(str(cm_path))
        axes[idx].imshow(img)
        axes[idx].set_title(report['model_name'], fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 7. Best Model Recommendation

In [ ]:
best = comparison_report['rankings'][0]
print('=' * 60)
print(f'🏆 RECOMMENDED MODEL: {best["model_name"].upper()}')
print('=' * 60)
print(f'  Composite Score: {best["composite_score"]:.4f}')
print(f'  Macro F1:        {best["raw_metrics"]["macro_f1"]:.4f}')
print(f'  Accuracy:        {best["raw_metrics"]["accuracy"]:.4f}')
print(f'  Latency:         {best["raw_metrics"]["latency_ms"]:.1f} ms')
print(f'  Model Size:      {best["raw_metrics"]["model_size_mb"]:.1f} MB')
print('\nCheckpoint path:')
print(f'  ml/artifacts/checkpoints/{best["model_name"]}_best.pth')
print('\nTo deploy this model, update the backend environment variable:')
print(f'  MODEL_PATH=ml/artifacts/checkpoints/{best["model_name"]}_best.pth')
print(f'  MODEL_NAME={best["model_name"]}')

## 8. Export for Deployment

Copy the best checkpoint to the `models/` directory for the backend to load.

In [ ]:
import shutil

best_name = best['model_name']
src = PROJECT_ROOT / 'ml' / 'artifacts' / 'checkpoints' / f'{best_name}_best.pth'
dst = PROJECT_ROOT / 'models' / f'{best_name}_best.pth'

if src.exists():
    shutil.copy2(src, dst)
    print(f'✓ Copied {src.name} → {dst}')

    # Also save class names for the backend
    import torch
    ckpt = torch.load(src, map_location='cpu', weights_only=False)
    if 'class_names' in ckpt:
        classes_path = PROJECT_ROOT / 'models' / 'classes.txt'
        with open(classes_path, 'w') as f:
            f.write(','.join(ckpt['class_names']))
        print(f'✓ Updated {classes_path}')
else:
    print(f'✗ Checkpoint not found: {src}')

---
**✅ Model comparison complete!**

### Next Steps
1. Start the backend: `PYTHONPATH=. uvicorn backend.app.main:app --reload`
2. Start the frontend: `cd frontend && npm run dev`
3. Upload images and verify predictions